##Loading and Pre-Processing


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import glob
import pandas as pd
import numpy as np

PROJECT_PATH = "/content/drive/MyDrive/Spring 2026/IDS_Project"

RAW_DATA = os.path.join(PROJECT_PATH, "raw_data")
ML_DATA = os.path.join(RAW_DATA, "MachineLearningCVE")
TRAFFIC_LABELING = os.path.join(RAW_DATA, "TrafficLabeling")

PROCESSED_DATA = os.path.join(PROJECT_PATH, "processed_data")
MODELS = os.path.join(PROJECT_PATH, "models")
RESULTS = os.path.join(PROJECT_PATH, "results")

print("Project path:", PROJECT_PATH)
print("ML path:", ML_DATA)
print("Traffic Labeling path:", TRAFFIC_LABELING)

Project path: /content/drive/MyDrive/Spring 2026/IDS_Project
ML path: /content/drive/MyDrive/Spring 2026/IDS_Project/raw_data/MachineLearningCVE
Traffic Labeling path: /content/drive/MyDrive/Spring 2026/IDS_Project/raw_data/TrafficLabeling


In [3]:
ml_files = sorted(glob.glob(os.path.join(ML_DATA, "*.csv")))

print("Number of MachineLearningCVE files:", len(ml_files))
for f in ml_files:
    print(os.path.basename(f))

Number of MachineLearningCVE files: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [4]:
dfs = []

for file in ml_files:
    print(f"Loading: {os.path.basename(file)}")
    df = pd.read_csv(file, low_memory=False)
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print("Combined shape:", data.shape)

Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv


OSError: [Errno 107] Transport endpoint is not connected

In [ ]:
data.columns = data.columns.str.strip()
print(data.columns.tolist())

In [ ]:
print("Label" in data.columns)

In [ ]:
data["Label"].value_counts()

In [ ]:
print("Missing values before cleaning:", data.isna().sum().sum())

numeric_cols = data.select_dtypes(include=[np.number]).columns
inf_count = np.isinf(data[numeric_cols]).sum().sum()

print("Infinite values before cleaning:", inf_count)

In [ ]:
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)

print("Shape after removing NaN and inf rows:", data.shape)

In [ ]:
duplicate_count = data.duplicated().sum()
print("Duplicate rows:", duplicate_count)

In [ ]:
data.drop_duplicates(inplace=True)
print("Shape after removing duplicates:", data.shape)

In [ ]:
data["BinaryLabel"] = data["Label"].apply(
    lambda x: 0 if str(x).strip() == "BENIGN" else 1
)

data["BinaryLabel"].value_counts()

In [ ]:
output_path = os.path.join(PROCESSED_DATA, "cicids2017_cleaned.csv")
#data.to_csv(output_path, index=False)

print("Saved cleaned dataset to:")
print(output_path)

In [ ]:
binary_output_path = os.path.join(PROCESSED_DATA, "cicids2017_binary.csv")

#binary_data = data.copy()
#binary_data = binary_data.drop(columns=["Label"])

#binary_data.to_csv(binary_output_path, index=False)

print("Saved binary dataset to:")
print(binary_output_path)

## Machine Learning

In [ ]:
import os
import pandas as pd
import numpy as np

PROJECT_PATH = "/content/drive/MyDrive/Spring 2026/IDS_Project"
PROCESSED_DATA = os.path.join(PROJECT_PATH, "processed_data")

clean_path = os.path.join(PROCESSED_DATA, "cicids2017_cleaned.csv")

data = pd.read_csv(clean_path, low_memory=False)
print("Loaded shape:", data.shape)
data.head()

In [ ]:
print(data.columns.tolist())

In [ ]:
target_col = "BinaryLabel"
drop_cols = ["Label", "BinaryLabel"]

X = data.drop(columns=drop_cols, errors="ignore")
y = data[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X = X.select_dtypes(include=[np.number])

print("Numeric X shape:", X.shape)

In [ ]:
constant_cols = [col for col in X.columns if X[col].nunique() <= 1]

print("Number of constant columns:", len(constant_cols))
print("Constant columns:", constant_cols)

X = X.drop(columns=constant_cols)

print("Shape after removing constant columns:", X.shape)

In [ ]:
corr_matrix = X.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_cols = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.95)]

print("Number of highly correlated columns to remove:", len(high_corr_cols))
print(high_corr_cols[:20])  # show first 20

X = X.drop(columns=high_corr_cols)

print("Shape after removing highly correlated columns:", X.shape)

In [ ]:
print(y.value_counts())
print(y.value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
prepared_train_path = os.path.join(PROCESSED_DATA, "X_train.csv")
prepared_test_path = os.path.join(PROCESSED_DATA, "X_test.csv")
y_train_path = os.path.join(PROCESSED_DATA, "y_train.csv")
y_test_path = os.path.join(PROCESSED_DATA, "y_test.csv")

pd.DataFrame(X_train, columns=X.columns).to_csv(prepared_train_path, index=False)
pd.DataFrame(X_test, columns=X.columns).to_csv(prepared_test_path, index=False)
pd.DataFrame(y_train, columns=["BinaryLabel"]).to_csv(y_train_path, index=False)
pd.DataFrame(y_test, columns=["BinaryLabel"]).to_csv(y_test_path, index=False)

print("Prepared datasets saved.")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import joblib

scaler_path = os.path.join(PROJECT_PATH, "models", "standard_scaler.pkl")
#joblib.dump(scaler, scaler_path)

print("Scaler saved to:", scaler_path)

In [ ]:
print("Final feature count:", X.shape[1])
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Class distribution:\n", y.value_counts())

## Logistic Regression Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

# Re-initialize and fit log_model to ensure it's defined
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
log_precision = precision_score(y_test, y_pred)
log_recall = recall_score(y_test, y_pred)
log_f1 = f1_score(y_test, y_pred)


print("Accuracy:", accuracy)
print("Precision:", log_precision)
print("Recall:", log_recall)
print("F1-score:", log_f1)

print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression Confusion Matrix")

plt.show()

In [ ]:
import joblib

model_path = os.path.join(PROJECT_PATH, "models", "logistic_regression_model.pkl")

#joblib.dump(log_model, model_path)

print("Model saved to:", model_path)

In [ ]:
print("Model: Logistic Regression")
print("Accuracy:", accuracy)

print("\nClass distribution:")
print(y.value_counts())

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
import time
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
rf_start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_end = time.time()
rf_train_time = rf_end - rf_start

print("Random Forest training time (seconds):", rf_train_time)

In [ ]:
rf_pred = rf_model.predict(X_test)

In [ ]:
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest Precision:", rf_precision)
print("Random Forest Recall:", rf_recall)
print("Random Forest F1-score:", rf_f1)

In [ ]:
print(classification_report(y_test, rf_pred))

In [ ]:
rf_cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(rf_cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
import joblib
import os

rf_model_path = os.path.join(PROJECT_PATH, "models", "random_forest_model.pkl")
joblib.dump(rf_model, rf_model_path)

print("Random Forest model saved to:", rf_model_path)

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy,
        "Precision": log_precision,
        "Recall": log_recall,
        "F1-Score": log_f1,
        "Train Time (s)": log_train_time
    },
    {
        "Model": "Random Forest",
        "Accuracy": rf_accuracy,
        "Precision": rf_precision,
        "Recall": rf_recall,
        "F1-Score": rf_f1,
        "Train Time (s)": rf_train_time
    }
])

print(comparison_df)

In [ ]:
comparison_path = os.path.join(RESULTS, "model_comparison_baseline.csv")
comparison_df.to_csv(comparison_path, index=False)

print("Comparison table saved to:", comparison_path)

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

print(feature_importance.head(20))

In [ ]:
top_n = 20
top_features = feature_importance.head(top_n)

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Random Forest Feature Importances")
plt.tight_layout()
plt.show()